# Лабораторная работа 4: регрессия k ближайших соседей на данных Boston

## 1. Загрузите выборку Boston с помощью функции sklearn.datasets.load_boston().
Результатом вызова данной функции является объект, у которого признаки записаны в поле data, а целевой вектор — в поле target.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import scale
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score
import seaborn as sb
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('boston.data')
data

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273.0,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273.0,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273.0,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273.0,21.0,393.45,6.48,22.0


Выделение признаков и целевой переменной.

In [3]:
X = data[data.columns[:-1]]
y = data['MEDV']

## 2. Приведите признаки в выборке к одному масштабу при помощи функции sklearn.preprocessing.scale.

In [4]:
X_scl = scale(X)

## 3. Переберите разные варианты параметра метрики p по сетке от 1 до 10 с таким шагом, чтобы всего было протестировано 200 вариантов (используйте функцию numpy.linspace). Используйте KNeighborsRegressor с n_neighbors=5 и weights='distance' — данный параметр добавляет в алгоритм веса, зависящие от расстояния до ближайших соседей. В качестве метрики качества используйте среднеквадратичную ошибку (параметр scoring='mean_squared_error' у cross_val_score; при использовании библиотеки scikit-learn версии 18.0.1 и выше необходимо указывать scoring='neg_mean_squared_error'). Качество оценивайте, как и в предыдущем задании, с помощью кросс-валидации по 5 блокам с random_state = 42, не забудьте включить перемешивание выборки (shuffle=True).

In [5]:
kf = KFold(shuffle=True, n_splits=5, random_state=42)

def evaluate_KNN(p, X, y):
    knn = KNeighborsRegressor(n_neighbors=5, weights='distance', p=p)
    return cross_val_score(knn, X, y, cv = kf, scoring='neg_mean_squared_error').mean()

eval_df = pd.DataFrame(columns=['p', 'eval_mean'])

for p in np.linspace(1,10, 200):
    eval_mean = evaluate_KNN(p, X_scl, y)
    eval_df.loc[len(eval_df)] = [p, eval_mean]
eval_df

,p,eval_mean
0,1.000000,-16.030647
1,1.045226,-16.407839
2,1.090452,-16.370697
3,1.135678,-16.445716
4,1.180905,-16.475058
...,...,...
195,9.819095,-21.081264
196,9.864322,-21.082127
197,9.909548,-21.082979
198,9.954774,-21.083819


## 4. Определите, при каком p качество на кросс-валидации оказалось оптимальным. Обратите внимание, что cross_val_score возвращает массив значений оценки качества для каждого блока, но если вы хотите получить среднее значение качества всей выборки, то следует использовать метод kfold(). Это значение параметра и будет ответом на задачу.

In [7]:
ans = eval_df.loc[eval_df['eval_mean'].idxmax(), 'p']

with open('1.txt', 'w') as file:
    file.write(f'{int(ans)}')